# Module 04.09 — Saved Queries (`query`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.9 Saved Queries (`query`)

A saved query stores a **reusable KQL or Lucene expression** — and optionally a set of
pinned filters and a time range — that appears in the search bar drop-down in Discover,
Dashboards, and the Alerting rule editor. Think of it as a named shortcut for a frequently
used search predicate.

Unlike a saved search, a saved query is *not* tied to a specific Data View. Its schema is
minimal: `title`, `description`, `query.query` (the expression string), `query.language`
(`kuery` or `lucene`), `filters[]`, and an optional `timefilter` with `from`/`to`/`refreshInterval`.
Because there is no `references` array, saved queries have no dependencies and are among
the simplest objects to restore.

The practical impact during a disaster-recovery scenario is that saved queries let your
analysts get back to work quickly — once Kibana is restored, every named search expression
they rely on in the search bar is immediately available again, without anyone needing to
remember and retype complex KQL predicates.

In [ ]:
heading('4.9 Saved Query — create')

sq_resp = kibana_post('/api/saved_queries/_create', {
    'title': 'High value orders',
    'description': 'Orders over $100',
    'query': {
        'query': 'taxful_total_price > 100',
        'language': 'kuery',
    },
    'filters': [],
    'timefilter': {
        'from': 'now-30d',
        'to': 'now',
        'refreshInterval': {'pause': True, 'value': 0},
    },
})
sq_id = sq_resp['id']
success(f'Saved query created: id={sq_id}')
pp(sq_resp, 'query saved object')
info('Key fields: title, description, query.query, query.language, filters[], timefilter')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
# Saved queries appear in the KQL search bar drop-down in Discover:
kibana_link('/app/discover', 'Discover — click the saved query icon in the search bar')

In [ ]:
# Snapshot → delete → restore
snap_delete_restore_cycle('snap-query-test', 'Saved Query')

requests_mod = __import__('requests')
requests_mod.delete(
    f'{KIBANA_HOST}/api/saved_queries/{sq_id}',
    auth=('elastic', ELASTIC_PASSWORD),
    headers={'kbn-xsrf': 'true'},
)
warn(f'Deleted saved query {sq_id}')

restore_kibana_state(es, SNAP_REPO, 'snap-query-test')
time.sleep(3)

queries_resp = kibana_get(f'/api/saved_queries/_all')
restored_ids = [q['id'] for q in queries_resp.get('savedQueries', [])]
assert sq_id in restored_ids
success('Saved query restored!')